# AutoResearch Layer 1: Training Experiments

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

# Install with Drive pip cache
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Clone/pull with retry
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed: {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed after 3 attempts')
    os.chdir(REPO_DIR)
print(f'Repo: {os.getcwd()}')

# Data
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
cases = [d for d in os.listdir(DATA_DIR) if d.startswith('BraTS')]
print(f'Data: {len(cases)} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')

Mounted at /content/drive
Sat Mar 28 13:48:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   28C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## L1-5: d_state=64 Quick Screen (30 epochs)\n\nQuick test: does Mamba2 with d_state=64 improve over d_state=16?

In [2]:
# L1-5: d_state=64 quick screen
import os, glob
os.chdir(REPO_DIR)

# Clean checkpoints
for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Cleaned checkpoints')

# Train from scratch, 30 epochs
!python -u train.py \
    --config configs/autoresearch/L1-5_dstate64_quick.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('L1-5')
print('L1-5 complete')

Cleaned checkpoints
2026-03-28 01:48:15.125045: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 01:48:15.441533: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774662495.464054  186602 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774662495.471595  186602 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774662495.491210  186602 computation_placer.cc:177] computation placer already registered. Please check linkag

In [3]:
# L1-5 Eval
import subprocess, re, os, json
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-5.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-5_dstate64_quick.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

Using: /content/drive/MyDrive/TextMamba3D/checkpoints/best_L1-5.pth
--- text+TTA ---
  dice_ET: 0.6770 +/- 0.2065
  dice_TC: 0.7894 +/- 0.1738
  dice_WT: 0.8506 +/- 0.0787
  dice_mean: 0.7723 +/- 0.1114
  hd95_ET: 5.44 +/- 13.76
  hd95_TC: 4.58 +/- 12.01
  hd95_WT: 4.80 +/- 14.90
--- notext+TTA ---
  dice_ET: 0.6870 +/- 0.1939
  dice_TC: 0.7745 +/- 0.1834
  dice_WT: 0.8449 +/- 0.0771
  dice_mean: 0.7688 +/- 0.1113
  hd95_ET: 5.29 +/- 12.63
  hd95_TC: 4.74 +/- 10.97
  hd95_WT: 4.87 +/- 13.42


## L1-1: Higher LR from Scratch (200 epochs, lr=1e-4)\n\nV5.0 used lr=5e-5. Double it to escape local minimum.

In [4]:
# L1-1: lr=1e-4 from scratch
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Cleaned checkpoints')

!python -u train.py \
    --config configs/autoresearch/L1-1_lr1e4_scratch.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('L1-1')
print('L1-1 complete')

Cleaned checkpoints
2026-03-28 03:34:09.034375: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 03:34:09.126732: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774668849.151881  219072 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774668849.159939  219072 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774668849.181262  219072 computation_placer.cc:177] computation placer already registered. Please check linkag

In [5]:
# L1-1 Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-1.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-1_lr1e4_scratch.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

Using: /content/drive/MyDrive/TextMamba3D/checkpoints/best_L1-1.pth
--- text+TTA ---
  dice_ET: 0.7869 +/- 0.2008
  dice_TC: 0.8511 +/- 0.1680
  dice_WT: 0.8957 +/- 0.0736
  dice_mean: 0.8446 +/- 0.1065
  hd95_ET: 2.70 +/- 6.12
  hd95_TC: 3.59 +/- 11.72
  hd95_WT: 2.60 +/- 10.92
--- notext+TTA ---
  dice_ET: 0.7738 +/- 0.2241
  dice_TC: 0.8492 +/- 0.1666
  dice_WT: 0.8957 +/- 0.0725
  dice_mean: 0.8395 +/- 0.1183
  hd95_ET: 2.46 +/- 4.68
  hd95_TC: 3.74 +/- 11.67
  hd95_WT: 2.68 +/- 10.58


## L1-4: ET-Weighted Training (200 epochs)\n\nET class weight 6.0 (vs default 4.0), early stopping on ET Dice.

In [6]:
# L1-4: ET-weighted
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

!python -u train.py \
    --config configs/autoresearch/L1-4_et_weighted.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2 \
    --es-metric et

sync_and_tag('L1-4')
print('L1-4 complete')

2026-03-28 07:59:33.066120: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 07:59:33.092879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774684773.117382  310497 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774684773.125416  310497 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774684773.148139  310497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [7]:
# L1-4 Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-4.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-4_et_weighted.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

Using: /content/drive/MyDrive/TextMamba3D/checkpoints/best_L1-4.pth
--- text+TTA ---
  dice_ET: 0.7806 +/- 0.2085
  dice_TC: 0.8517 +/- 0.1562
  dice_WT: 0.8864 +/- 0.0747
  dice_mean: 0.8396 +/- 0.1120
  hd95_ET: 4.68 +/- 13.70
  hd95_TC: 3.93 +/- 12.12
  hd95_WT: 4.16 +/- 15.15
--- notext+TTA ---
  dice_ET: 0.7770 +/- 0.2085
  dice_TC: 0.8412 +/- 0.1620
  dice_WT: 0.8826 +/- 0.0739
  dice_mean: 0.8336 +/- 0.1123
  hd95_ET: 4.68 +/- 13.60
  hd95_TC: 4.20 +/- 12.12
  hd95_WT: 4.24 +/- 15.17


## L1-3: Aggressive Augmentation (200 epochs)\n\nCopy-paste=0.5, no_text_ratio=0.3, all augmentations on.

In [ ]:
# L1-3: Aggressive augmentation
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

!python -u train.py \
    --config configs/autoresearch/L1-3_aggressive_aug.yaml \
    --grad-accum 2

sync_and_tag('L1-3')
print('L1-3 complete')

2026-03-28 11:39:21.918036: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 11:39:21.943650: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774697961.968464  385126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774697961.976519  385126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774697962.000191  385126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# Resume L1-3 from last checkpoint (epoch 60+)
import os, glob, shutil
os.chdir(REPO_DIR)

last_ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(last_ckpt), 'No last.pth found'
print(f'Resuming from: {last_ckpt}')

!python -u train.py \
    --config configs/autoresearch/L1-3_aggressive_aug.yaml \
    --resume "{last_ckpt}" \
    --grad-accum 2

sync_and_tag('L1-3')
print('L1-3 complete')

Resuming from: /content/drive/MyDrive/TextMamba3D/checkpoints/last.pth
2026-03-28 13:56:07.374127: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 13:56:07.393156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774706167.415781    7472 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774706167.423186    7472 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774706167.442427    7472 computation_placer.cc:177] computa

In [3]:
# L1-3 Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-3.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-3_aggressive_aug.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

Using: /content/drive/MyDrive/TextMamba3D/checkpoints/last.pth
--- text+TTA ---
  dice_ET: 0.7706 +/- 0.2062
  dice_TC: 0.8333 +/- 0.1781
  dice_WT: 0.8724 +/- 0.0827
  dice_mean: 0.8255 +/- 0.1185
  hd95_ET: 4.50 +/- 13.39
  hd95_TC: 4.26 +/- 12.28
  hd95_WT: 4.59 +/- 15.75
--- notext+TTA ---
  dice_ET: 0.7696 +/- 0.2080
  dice_TC: 0.8323 +/- 0.1795
  dice_WT: 0.8671 +/- 0.0838
  dice_mean: 0.8230 +/- 0.1197
  hd95_ET: 4.52 +/- 13.43
  hd95_TC: 4.27 +/- 12.26
  hd95_WT: 4.71 +/- 15.54


## L1 Summary

In [4]:
# L1 Results Summary
print('AutoResearch Layer 1 Results')
print('=' * 70)
print(f'Baseline V5.0: Mean=0.8479, ET=0.7910, TC=0.8560, WT=0.8967')
print()
print('Check eval cells above for each experiment result.')
print('Record the best result:')
print('  python -m autoresearch record L1-X \'{"dice_mean": 0.XX, "dice_ET": 0.XX}\'')

AutoResearch Layer 1 Results
Baseline V5.0: Mean=0.8479, ET=0.7910, TC=0.8560, WT=0.8967

Check eval cells above for each experiment result.
Record the best result:
  python -m autoresearch record L1-X '{"dice_mean": 0.XX, "dice_ET": 0.XX}'
